# NLP Lab 7 — TF-IDF, BM25


In [1]:
import nltk
import numpy as np
import math
from collections import defaultdict, Counter
from nltk.corpus import reuters
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from rank_bm25 import BM25Okapi
import warnings
warnings.filterwarnings('ignore')

for resource in ['reuters', 'punkt', 'punkt_tab', 'stopwords']:
    nltk.download(resource, quiet=True)

print("Всі залежності завантажено")
print(f"Reuters corpus: {len(reuters.fileids())} документів")

Всі залежності завантажено
Reuters corpus: 10788 документів


[nltk_data] Error loading reuters: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt_tab: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


---
## Task 1 — Bag-of-Words Tagger

Беремо довільний текст з NLTK corpora (reuters) і реалізуємо BoW tagger.

In [2]:
STOP_WORDS = set(stopwords.words('english'))

def tokenize(text: str, remove_stopwords: bool = True) -> list[str]:
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS]
    return tokens

class BagOfWords:

    def __init__(self, remove_stopwords: bool = True):
        self.remove_stopwords = remove_stopwords
        self.vocab: dict[str, int] = {}   # слово → індекс

    def fit(self, corpus: list[str]) -> "BagOfWords":
        all_tokens: set[str] = set()
        for doc in corpus:
            all_tokens.update(tokenize(doc, self.remove_stopwords))
        self.vocab = {word: idx for idx, word in enumerate(sorted(all_tokens))}
        print(f"Словник побудовано: {len(self.vocab)} унікальних термів")
        return self

    def transform(self, doc: str) -> np.ndarray:
        vec = np.zeros(len(self.vocab), dtype=np.int32)
        for token in tokenize(doc, self.remove_stopwords):
            if token in self.vocab:
                vec[self.vocab[token]] += 1
        return vec

    def fit_transform(self, corpus: list[str]) -> list[np.ndarray]:
        self.fit(corpus)
        return [self.transform(doc) for doc in corpus]

    def similarity(self, doc_a: str, doc_b: str) -> float:
        va, vb = self.transform(doc_a), self.transform(doc_b)
        denom = (np.linalg.norm(va) * np.linalg.norm(vb))
        return float(np.dot(va, vb) / denom) if denom > 0 else 0.0

    def top_terms(self, doc: str, n: int = 10) -> list[tuple[str, int]]:
        vec = self.transform(doc)
        idx_sorted = np.argsort(vec)[::-1][:n]
        idx_to_word = {v: k for k, v in self.vocab.items()}
        return [(idx_to_word[i], int(vec[i])) for i in idx_sorted if vec[i] > 0]

file_ids = reuters.fileids()[:200]
corpus_texts = [reuters.raw(fid) for fid in file_ids]

bow = BagOfWords(remove_stopwords=True)
bow.fit(corpus_texts)

demo_doc = reuters.raw(file_ids[0])
print(f"\nДокумент: {file_ids[0]}")
print(f"Текст (початок): {demo_doc[:150].strip()}")
print(f"\nTopk-10 термів у документі:")
for term, cnt in bow.top_terms(demo_doc):
    print(f"  {term:20s} {cnt}")

Словник побудовано: 3708 унікальних термів

Документ: test/14826
Текст (початок): ASIAN EXPORTERS FEAR DAMAGE FROM U.S.-JAPAN RIFT
  Mounting trade friction between the
  U.S. And Japan has raised fears among many of Asia's exportin

Topk-10 термів у документі:
  said                 16
  trade                15
  japan                12
  exports              6
  dlrs                 6
  tariffs              5
  imports              5
  billion              5
  would                4
  taiwan               4


In [3]:
doc_a = reuters.raw(file_ids[0])
doc_b = reuters.raw(file_ids[1])
doc_c = reuters.raw(file_ids[100])

sim_ab = bow.similarity(doc_a, doc_b)
sim_ac = bow.similarity(doc_a, doc_c)

print(f"Косинусна подібність між doc[0] і doc[1]   : {sim_ab:.4f}")
print(f"Косинусна подібність між doc[0] і doc[100] : {sim_ac:.4f}")
print("(очікуємо, що суміжні документи більш схожі)")
vec = bow.transform(doc_a)
print(f"\nРозмірність вектора: {len(vec)} (= розмір словника)")
print(f"Ненульових компонент: {np.count_nonzero(vec)}")

Косинусна подібність між doc[0] і doc[1]   : 0.1678
Косинусна подібність між doc[0] і doc[100] : 0.0794
(очікуємо, що суміжні документи більш схожі)

Розмірність вектора: 3708 (= розмір словника)
Ненульових компонент: 274


---
## Task 2 — N-gram Bag-of-Words Tagger

Розширюємо BoW до N-грамів: замість окремих слів використовуємо послідовності з N токенів.

In [4]:
class NGramBagOfWords:

    def __init__(self, n: int | list[int] = 2, remove_stopwords: bool = False):
        self.n_values = [n] if isinstance(n, int) else n
        self.remove_stopwords = remove_stopwords
        self.vocab: dict[tuple, int] = {}

    def _extract_ngrams(self, text: str) -> list[tuple[str, ...]]:
        tokens = tokenize(text, self.remove_stopwords)
        ngrams = []
        for n in self.n_values:
            ngrams.extend(
                tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)
            )
        return ngrams

    def fit(self, corpus: list[str]) -> "NGramBagOfWords":
        all_ngrams: set[tuple] = set()
        for doc in corpus:
            all_ngrams.update(self._extract_ngrams(doc))
        self.vocab = {ng: idx for idx, ng in enumerate(sorted(all_ngrams))}
        print(f"N-gram словник: {len(self.vocab)} унікальних {self.n_values}-грамів")
        return self

    def transform(self, doc: str) -> np.ndarray:
        vec = np.zeros(len(self.vocab), dtype=np.int32)
        for ng in self._extract_ngrams(doc):
            if ng in self.vocab:
                vec[self.vocab[ng]] += 1
        return vec

    def fit_transform(self, corpus: list[str]) -> list[np.ndarray]:
        self.fit(corpus)
        return [self.transform(doc) for doc in corpus]

    def similarity(self, doc_a: str, doc_b: str) -> float:
        va, vb = self.transform(doc_a), self.transform(doc_b)
        denom = np.linalg.norm(va) * np.linalg.norm(vb)
        return float(np.dot(va, vb) / denom) if denom > 0 else 0.0

    def top_ngrams(self, doc: str, n: int = 10) -> list[tuple[tuple, int]]:
        vec = self.transform(doc)
        idx_sorted = np.argsort(vec)[::-1][:n]
        idx_to_ng = {v: k for k, v in self.vocab.items()}
        return [(idx_to_ng[i], int(vec[i])) for i in idx_sorted if vec[i] > 0]

ngbow_12 = NGramBagOfWords(n=[1, 2], remove_stopwords=False)
ngbow_12.fit(corpus_texts)

print(f"\nТоп-10 n-грамів у doc[0]:")
for ng, cnt in ngbow_12.top_ngrams(doc_a):
    print(f"  {str(ng):40s} {cnt}")

N-gram словник: 19768 унікальних [1, 2]-грамів

Топ-10 n-грамів у doc[0]:
  ('the',)                                 37
  ('of',)                                  30
  ('to',)                                  26
  ('and',)                                 16
  ('said',)                                16
  ('in',)                                  16
  ('a',)                                   15
  ('trade',)                               15
  ('japan',)                               12
  ('for',)                                 7


In [5]:
print("Порівняння розміру словника для різних n:")
for n_val in [1, 2, 3]:
    tagger = NGramBagOfWords(n=n_val, remove_stopwords=False)
    tagger.fit(corpus_texts)
    sim = tagger.similarity(doc_a, doc_b)
    print(f"  n={n_val}: vocab={len(tagger.vocab):>7}, sim(doc0, doc1)={sim:.4f}")

print("\nВисновок: зі зростанням n словник зростає, але подібність падає —")
print("n-грами точніше описують текст, але рідше збігаються між документами.")

Порівняння розміру словника для різних n:
N-gram словник: 3820 унікальних [1]-грамів
  n=1: vocab=   3820, sim(doc0, doc1)=0.5862
N-gram словник: 15948 унікальних [2]-грамів
  n=2: vocab=  15948, sim(doc0, doc1)=0.0310
N-gram словник: 21508 унікальних [3]-грамів
  n=3: vocab=  21508, sim(doc0, doc1)=0.0000

Висновок: зі зростанням n словник зростає, але подібність падає —
n-грами точніше описують текст, але рідше збігаються між документами.


---
## Task 3 — Information Retrieval (BM25) + Keyword Summarization (TF-IDF)

Датасет: **TED Talk Transcripts 2006–2021** (Kaggle).  

In [6]:
import pandas as pd
import numpy as np
import math
import warnings
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
warnings.filterwarnings('ignore')

for r in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(r, quiet=True)

df = pd.read_csv('transcript_data.csv') 
df = df[df['transcript'].str.strip().astype(bool)].reset_index(drop=True)

print(f'Документів у датасеті : {len(df)}')
print(f'Колонки               : {list(df.columns)}')
print(f'Середня довжина току  : {df["transcript"].str.split().str.len().mean():.0f} слів')
df.head(3)


[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt_tab: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


Документів у датасеті : 4442
Колонки               : ['title', 'transcript']
Середня довжина току  : 1736 слів


,title,transcript
0,Can you outsmart the apples and oranges fallacy?,Baking apple pie? Discount orange warehouse ha...
1,The exploitation of US college athletes,"In college sports, American universities are e..."
2,How does ultrasound work?,"In a pitch-black cave, bats can’t see much. Bu..."


In [7]:
STOP_WORDS = set(stopwords.words('english'))

def tokenize(text: str, remove_stopwords: bool = True) -> list:
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha()]
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS]
    return tokens


class TFIDF:

    def __init__(self):
        self.vocab = {}
        self.idf = np.array([])
        self.N = 0

    def fit(self, corpus: list):
        self.N = len(corpus)
        all_tokens = set()
        for doc in corpus:
            all_tokens.update(tokenize(doc))
        self.vocab = {w: i for i, w in enumerate(sorted(all_tokens))}

        df = np.zeros(len(self.vocab), dtype=np.float64)
        for doc in corpus:
            for t in set(tokenize(doc)):
                if t in self.vocab:
                    df[self.vocab[t]] += 1

        self.idf = np.log((self.N + 1) / (df + 1))
        print(f'TFIDF fit: vocab={len(self.vocab):,}, docs={self.N:,}')
        return self

    def _tf(self, tokens: list) -> np.ndarray:
        cnt = Counter(tokens)
        tf = np.zeros(len(self.vocab), dtype=np.float64)
        for t, c in cnt.items():
            if t in self.vocab:
                tf[self.vocab[t]] = 1 + math.log(c)
        return tf

    def transform(self, doc: str) -> np.ndarray:
        return self._tf(tokenize(doc)) * self.idf

    def keywords(self, doc: str, top_n: int = 10) -> list:
        vec = self.transform(doc)
        idx_to_word = {v: k for k, v in self.vocab.items()}
        top_idx = np.argsort(vec)[::-1][:top_n]
        return [(idx_to_word[i], round(float(vec[i]), 4))
                for i in top_idx if vec[i] > 0]


# Fit на повному корпусі
corpus = df['transcript'].tolist()
tfidf = TFIDF()
tfidf.fit(corpus)


TFIDF fit: vocab=67,653, docs=4,442


In [8]:
print('=' * 65)
print('KEYWORD SUMMARIZATION — TED Talks (TF-IDF)')
print('=' * 65)

samples = df.iloc[[0, 50, 200, 500, 1000]]

for _, row in samples.iterrows():
    keywords = tfidf.keywords(row['transcript'], top_n=8)
    print(f'\n Talk    : {row["title"]}')
    print(f' Snippet : {str(row["transcript"])[:120].strip()}...')
    print(f' Keywords: {[w for w, _ in keywords]}')


KEYWORD SUMMARIZATION — TED Talks (TF-IDF)

 Talk    : Can you outsmart the apples and oranges fallacy?
 Snippet : Baking apple pie? Discount orange warehouse has you covered! A fruit’s a fruit, right?It’s 1988, and scientist James Han...
 Keywords: ['insufferable', 'warming', 'analogy', 'oranges', 'fever', 'witnesses', 'false', 'apples']

 Talk    : I let algorithms randomize my life for two years
 Snippet : I used to love waking up at exactly 7:00am. When I lived in San Francisco, I would wake up right at 7:00am and immediate...
 Keywords: ['random', 'generator', 'mumbai', 'yoga', 'francisco', 'uber', 'preference', 'tidy']

 Talk    : Why is 1.5 degrees such a big deal?
 Snippet : Why is 1.5 degrees such a big deal? Because to warm our entire planet up by 1.5 degrees Celsius requires a lot of heat.A...
 Keywords: ['intensifies', 'degrees', 'warmest', 'ferocity', 'wetter', 'drier', 'flourished', 'coldest']

 Talk    : For the love of fangirls
 Snippet : Four years ago, a teenage girl 

In [10]:
class BM25:


    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b  = b

    def fit(self, corpus: list):
        self.N = len(corpus)
        self.tokenized_docs = [tokenize(doc) for doc in corpus]
        self.doc_freqs = [Counter(t) for t in self.tokenized_docs]
        self.avgdl = np.mean([len(t) for t in self.tokenized_docs])

        self.df = {}
        for tokens in self.tokenized_docs:
            for t in set(tokens):
                self.df[t] = self.df.get(t, 0) + 1

        self.idf = {
            t: math.log((self.N - df + 0.5) / (df + 0.5))
            for t, df in self.df.items()
        }
        print(f'BM25 fit: {self.N:,} docs, avgdl={self.avgdl:.0f} tokens, k1={self.k1}, b={self.b}')
        return self

    def score(self, query_tokens: list, doc_idx: int) -> float:
        cnt = self.doc_freqs[doc_idx]
        doc_len = len(self.tokenized_docs[doc_idx])
        s = 0.0
        for q in query_tokens:
            if q not in self.idf:
                continue
            tf = cnt.get(q, 0)
            num = tf * (self.k1 + 1)
            den = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            s += self.idf[q] * num / den
        return s

    def retrieve(self, query: str, top_n: int = 5) -> list:
        q_tokens = tokenize(query)
        scores = np.array([self.score(q_tokens, i) for i in range(self.N)])
        top_idx = np.argsort(scores)[::-1][:top_n]
        return [(int(i), round(float(scores[i]), 4)) for i in top_idx]


bm25 = BM25(k1=1.5, b=0.75)
bm25.fit(corpus)


BM25 fit: 4,442 docs, avgdl=776 tokens, k1=1.5, b=0.75


In [11]:
queries = [
    'artificial intelligence machine learning future',
    'climate change ocean temperature',
    'human brain neuroscience memory',
    'democracy voting elections freedom',
]

print('=' * 65)
print('INFORMATION RETRIEVAL — TED Talks (BM25)')
print('=' * 65)

for query in queries:
    results = bm25.retrieve(query, top_n=3)
    print(f'\nQuery: "{query}"')
    print('-' * 65)
    for rank, (idx, score) in enumerate(results, 1):
        title   = df.loc[idx, 'title']
        snippet = str(df.loc[idx, 'transcript'])[:100].replace('\n', ' ')
        print(f'  #{rank} [{score:>6.2f}] {title}')
        print(f'        {snippet}...')


INFORMATION RETRIEVAL — TED Talks (BM25)

Query: "artificial intelligence machine learning future"
-----------------------------------------------------------------
  #1 [ 18.71] How does artificial intelligence learn?
        Today, artificial intelligence helps doctors diagnose patients, pilots fly commercial aircraft,  and...
  #2 [ 18.07] Art in the age of machine intelligence
        Hi, I'm Refik. I'm a media artist. I use data as a pigment and paint with a thinking brush that is a...
  #3 [ 17.92] What happens when our computers get smarter than we are?
        I work with a bunch of mathematicians, philosophers and computer scientists, and we sit around and t...

Query: "climate change ocean temperature"
-----------------------------------------------------------------
  #1 [ 15.23] Discovering ancient climates in oceans and ice
        If you really want to understand the problem that we're facing with the oceans, you have to think ab...
  #2 [ 15.04] The Arctic vs. the Antarc

In [12]:
query = 'space exploration Mars rockets'
results = bm25.retrieve(query, top_n=5)

print(f'Query: "{query}"\n')
print(f'{"#":<3} {"Score":>6}  {"Title":<48}  Top Keywords')
print('-' * 100)

for rank, (idx, score) in enumerate(results, 1):
    title    = df.loc[idx, 'title'][:46]
    keywords = [w for w, _ in tfidf.keywords(df.loc[idx, 'transcript'], top_n=5)]
    print(f'{rank:<3} {score:>6.2f}  {title:<48}  {keywords}')


Query: "space exploration Mars rockets"

#    Score  Title                                             Top Keywords
----------------------------------------------------------------------------------------------------
1    18.08  SpaceX's plan to fly you across the globe in 3    ['bfr', 'gs', 'falcon', 'gwynne', 'elon']
2    17.48  Your kids might live on Mars. Here's how they'    ['mars', 'spacex', 'rocket', 'elon', 'musk']
3    16.84  Let's not use Mars as a backup planet             ['habitable', 'interplanetary', 'habitability', 'fermi', 'kepler']
4    15.39  Model rocketry                                    ['supersonic', 'rockets', 'rocket', 'propellant', 'onboard']
5    14.75  Small rockets are the next space revolution       ['rocket', 'turbo', 'orbit', 'spacecraft', 'rockets']
